<a href="https://colab.research.google.com/github/ProfAndersonVanin/IBM3130-PLN-2026/blob/main/semana-09/aula09_EXEMPLO_NER_E_ANALISE_SENTIMENTOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Exemplo prático**
Vamos analisar:

`"A empresa Samsung lançou um novo celular em São Paulo. O aparelho tem uma câmera excelente, mas a bateria é decepcionante."`

Queremos descobrir:
1. Quais são as entidades?
2. Qual é o sentimento geral do texto?
3. Qual entidade está associada a uma opinião positiva?
4. Qual entidade está associada a uma opinião negativa?

# **1. Instalação**
Podemos usar spaCy para NER e TextBlob para uma primeira demonstração de análise de sentimentos.

In [ ]:
!pip install -q spacy textblob
!python -m spacy download pt_core_news_sm

# **2. Importação**

In [ ]:
import spacy
from textblob import TextBlob
nlp = spacy.load("pt_core_news_sm")

# **3. Texto**

In [ ]:
texto = """
A empresa Samsung lançou um novo celular em São Paulo.
O aparelho tem uma câmera excelente, mas a bateria é decepcionante.
"""

# **4. Aplicando NER**

In [ ]:
doc = nlp(texto)

print("ENTIDADES ENCONTRADAS")
print("=" * 40)

for entidade in doc.ents:
    print(entidade.text, "->", entidade.label_)

Podemos ter algo semelhante a:

```
Samsung -> ORG
São Paulo -> LOC
```

Dependendo do modelo, o resultado pode variar.



# **5. Visualizando o NER**

In [ ]:
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

# **6. Análise de sentimentos**
Aqui temos uma questão didática importante.

O `TextBlob` é muito simples e **não é uma solução adequada para análise de sentimentos em português**. Ele funciona melhor em inglês.

Por isso, para uma aula em português, podemos fazer primeiro uma demonstração conceitual e depois apresentar um modelo específico para português.

Para mostrar a lógica da análise, podemos usar:

In [ ]:
texto_ingles = """
Samsung launched a new smartphone.
The camera is excellent, but the battery is disappointing.
"""

blob = TextBlob(texto_ingles)

print("Polaridade:", blob.sentiment.polarity)

A polaridade varia aproximadamente entre:


```
-1 → muito negativo
 0 → neutro
+1 → muito positivo
```



# **7. Usando um modelo de sentimentos em português**
Para uma aula mais adequada ao conteúdo de PLN, vamos usar um modelo baseado em Transformers.

## Instalação:

In [ ]:
!pip install -q transformers torch

In [ ]:
from transformers import pipeline

Podemos carregar um modelo de análise de sentimentos em português:

In [ ]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="pysentimiento/bertweet-pt-sentiment"
)

E analisar:

In [ ]:
texto

In [ ]:
resultado = sentiment_pipeline(texto)

print(resultado)

O modelo pode retornar algo semelhante a:


```
[
    {'label': 'POS', 'score': 0.82}
]
```

O score representa a confiança do modelo.


# **8. O mais interessante: combinar NER + sentimento**
Agora podemos mostrar que as duas técnicas podem trabalhar juntas.

Imagine este texto:

In [ ]:
texto = """
A Samsung lançou um novo celular em São Paulo.
A câmera é excelente, mas a bateria é decepcionante.
A Apple apresentou um aparelho muito bonito e rápido.
"""

Primeiro fazemos NER:

In [ ]:
doc = nlp(texto)

print("ENTIDADES")

for entidade in doc.ents:
    print(entidade.text, "->", entidade.label_)

Depois fazemos análise de sentimento:

In [ ]:
resultado = sentiment_pipeline(texto)

print("\nSENTIMENTO")

print(resultado)

Teremos então duas análises diferentes sobre o mesmo texto:

```
NER:

Samsung → ORG
São Paulo → LOC
Apple → ORG
```
E

```
Sentimento:

POS / NEG / NEU
```




# **9. Indo além: sentimento por frase**
Aqui está uma aplicação mais interessante.

Em vez de analisar o texto inteiro, podemos separar as frases:

In [ ]:
frases = [
    "A câmera é excelente.",
    "A bateria é decepcionante.",
    "O aparelho é bonito e rápido."
]

for frase in frases:

    resultado = sentiment_pipeline(frase)

    print(frase)
    print(resultado)
    print()

Conceitualmente:

```
A câmera é excelente.
→ POSITIVO

A bateria é decepcionante.
→ NEGATIVO

O aparelho é bonito e rápido.
→ POSITIVO
```



# **10. Criando uma estrutura de dados**
Aqui começa uma parte muito interessante. Podemos transformar o resultado em uma estrutura:

In [ ]:
dados = [
    {
        "entidade": "Samsung",
        "tipo": "ORG",
        "sentimento": "positivo"
    },
    {
        "entidade": "Apple",
        "tipo": "ORG",
        "sentimento": "positivo"
    }
]

dados

Isso nos leva para uma ideia importante:
> **NER identifica "sobre quem ou sobre o quê" estamos falando.**

> **Análise de sentimentos identifica "qual é a opinião" sobre aquilo.**

Essa combinação é muito utilizada em análise de avaliações, redes sociais, notícias e monitoramento de marcas.

# **11. Exemplo mais próximo de uma aplicação real**

In [ ]:
texto = """
A Samsung apresentou um celular excelente.
A câmera possui ótima qualidade e o design é bonito.
Porém, a bateria é ruim e o preço é muito alto.
A Apple continua oferecendo aparelhos rápidos e confiáveis.
"""

doc = nlp(texto)

print("ENTIDADES")
print("=" * 40)

for entidade in doc.ents:
    print(entidade.text, "->", entidade.label_)

In [ ]:
print("\nSENTIMENTO POR FRASE")
print("=" * 40)

for frase in texto.split("."):

    frase = frase.strip()

    if frase:

        resultado = sentiment_pipeline(frase)

        print("\nFrase:", frase)
        print("Sentimento:", resultado)

O resultado pode ser interpretado aproximadamente assim:


| Texto | Entidade | Sentimento |
| :--- | :---: | ---: |
| Samsung apresentou um celular excelente | Samsung | Positivo |
| A câmera possui ótima qualidade | Samsung/celular | Positivo |
| o design é bonito | Samsung/celular | Positivo |
| a bateria é ruim | Samsung/celular | Negativo |
| o preço é muito alto | Samsung/celular | Negativo |
| Apple oferece aparelhos rápidos | Apple | Positivo |
| aparelhos confiáveis | Apple | Positivo |


## **E aqui surge um conceito mais avançado**
Isso nos leva ao **Aspect-Based Sentiment Analysis (ABSA)**, ou Análise de Sentimentos Baseada em Aspectos.

Em vez de simplesmente perguntar:
> "O texto é positivo ou negativo?"

perguntamos:
> "Qual é o sentimento em relação a cada aspecto?"

Por exemplo:

```
Samsung
 ├── câmera → positivo
 ├── design → positivo
 ├── bateria → negativo
 └── preço → negativo
```



# **Desafio: "Qual smartphone comprar?"**

Cada grupo recebe 10 ou 15 avaliações de smartphones.

O programa deve:
1. Identificar as empresas com NER.
2. Identificar os aspectos mencionados.
3. Classificar o sentimento.
4. Contabilizar sentimentos positivos e negativos.
5. Criar um ranking.
6. Apresentar uma recomendação final.
No final, o sistema poderia produzir:

```
==================================
ANÁLISE DE SMARTPHONES
==================================

🥇 Samsung
Positivos: 8
Negativos: 3

🥈 Apple
Positivos: 7
Negativos: 4

🥉 Xiaomi
Positivos: 6
Negativos: 5

==================================
ASPECTOS MAIS ELOGIADOS
==================================

Câmera
Desempenho
Design

==================================
ASPECTOS MAIS CRITICADOS
==================================

Preço
Bateria
```



# **30 avaliações de smartphones**
1. A câmera do Samsung Galaxy S25 é excelente e produz fotos muito nítidas.
2. O iPhone 17 tem um desempenho fantástico, mas achei o preço muito alto.
3. O Xiaomi Redmi Note 14 apresenta uma bateria excelente e dura o dia inteiro.
4. A câmera do iPhone 17 é ótima, principalmente em ambientes com pouca iluminação.
5. O Samsung Galaxy S25 é rápido, bonito e possui uma tela de excelente qualidade.
6. Achei a bateria do iPhone 17 decepcionante para um celular tão caro.
7. O Xiaomi Redmi Note 14 tem um preço muito interessante, mas a câmera poderia ser melhor.
8. A tela do Samsung Galaxy S25 é fantástica, com cores muito bonitas e excelente brilho.
9. O iPhone 17 é muito rápido e o sistema funciona de maneira bastante fluida.
10. O Redmi Note 14 esquenta bastante durante os jogos, e isso me incomodou.
11. Comprei o Samsung Galaxy S25 pela câmera e não me arrependi, as fotos são excelentes.
12. O preço do iPhone 17 é absurdo, apesar do excelente desempenho.
13. A bateria do Xiaomi Redmi Note 14 é um dos pontos mais positivos do aparelho.
14. O design do iPhone 17 é elegante, mas o aparelho é muito pesado.
15. O Samsung Galaxy S25 apresenta ótimo desempenho nos jogos e não trava.
16. A câmera frontal do Redmi Note 14 não me agradou, as selfies ficaram com pouca qualidade.
17. O iPhone 17 possui uma tela excelente e uma ótima qualidade de construção.
18. O Samsung Galaxy S25 tem uma bateria razoável, mas esperava uma autonomia maior.
19. O Redmi Note 14 é um celular barato e oferece um desempenho surpreendente pelo preço.
20. Fiquei decepcionado com o armazenamento do iPhone 17, que achei insuficiente.
21. A câmera do Samsung Galaxy S25 é melhor do que eu esperava e funciona muito bem à noite.
22. O Xiaomi Redmi Note 14 tem uma tela bonita, mas o brilho poderia ser maior.
23. O iPhone 17 é caro, porém entrega excelente desempenho e ótimas fotos.
24. O Samsung Galaxy S25 é bonito, rápido e confortável de usar durante o dia.
25. A bateria do Redmi Note 14 é excelente, mas o aparelho demora muito para carregar.
26. Não gostei do preço do Samsung Galaxy S25, mas reconheço que o desempenho é muito bom.
27. O iPhone 17 faz vídeos incríveis e possui uma câmera muito confiável.
28. O Redmi Note 14 apresenta bom desempenho para tarefas básicas, mas não é ideal para jogos pesados.
29. O Samsung Galaxy S25 possui uma tela maravilhosa, mas achei o aparelho um pouco caro.
30. Estou muito satisfeito com o iPhone 17, principalmente pela câmera, pelo desempenho e pela qualidade da tela.